# Ark+ Backbone Verification

**Model**: Ark+ (`jlianglab/Ark`) — Swin Transformer Large, cyclic multi-task pretraining  
**Arch**: bare `timm.models.swin_transformer.SwinTransformer` (Swin-L/192 @ 768px, window 12)  
**Embed dim**: 1536 (raw Swin-L `forward_features`)  
**Input**: 3×768×768 px, ImageNet normalization  
**Access**: Weights require form request (Google Form, see below) — **not auto-downloadable**  
**Zero-shot**: No — vision-only

Paper: [A Fully Open AI Foundation Model Applied to Chest Radiography (Nature, 2025)](https://www.nature.com/articles/s41586-025-09079-8)

### Prerequisites

```bash
# 1. Clone the repo
git clone https://github.com/jlianglab/Ark ../../third_party_models/Ark

# 2. Request pretrained weights:
#    Google Form (international): https://forms.gle/qkoDGXNiKRPTDdCe8
#    WeChat (China): https://www.wjx.cn/vm/OvwfYFx.aspx
#    Checkpoint filename: Ark6_swinLarge768_ep50.pth.tar
#    Place at: ../../third_party_models/Ark/Ark_Plus/Ark6_swinLarge768_ep50.pth.tar
```

> ⚠️ **timm version conflict**: Ark+'s checkpoint was saved with `timm==0.5.4`, which is
> incompatible with EVA-X (`timm==0.9.0`) and the rest of the project. The load cell
> below installs `timm 0.5.4` into a side-by-side directory
> (`third_party_models/timm-054`) and prepends it to `sys.path` for this notebook
> only — your main venv's `timm` is untouched.

In [ ]:
# %pip install -q "timm==0.5.4" torch torchvision Pillow

import subprocess, os, sys

ARK_DIR = "../../third_party_models/Ark/"
if not os.path.exists(ARK_DIR):
    subprocess.run(["git", "clone", "https://github.com/jlianglab/Ark", ARK_DIR], check=True)
    print("Cloned Ark")
else:
    print("Ark already present")

# The OmniSwinTransformer class lives in Ark_Plus/
ARK_PLUS_DIR = os.path.join(ARK_DIR, "Ark_Plus")
sys.path.insert(0, ARK_PLUS_DIR)
print(os.listdir(ARK_PLUS_DIR))

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Locate the checkpoint

In [ ]:
# Update this path after receiving the checkpoint
CKPT_PATH = "../../third_party_models/Ark/Ark_Plus/Ark6_swinLarge768_ep50.pth.tar"

assert os.path.exists(CKPT_PATH), (
    f"Checkpoint not found at {CKPT_PATH}.\n"
    "Request at: https://forms.gle/qkoDGXNiKRPTDdCe8"
)
print("Checkpoint found:", os.path.getsize(CKPT_PATH) / 1e6, "MB")

## 2. Inspect checkpoint structure

In [ ]:
import torch
import numpy as np

# PyTorch 2.6 defaults to weights_only=True; legacy pickles from
# NumPy 1.x checkpoints need these globals on the allowlist. The tuple
# form `(obj, "legacy_name")` lets us register the NumPy 2.x object under
# its old `numpy.core.*` import path (NumPy 2.x renamed `numpy.core`
# → `numpy._core`, so the pickle stream and the registered global no
# longer share a `__module__`).
torch.serialization.add_safe_globals([
    (np.core.multiarray.scalar, "numpy.core.multiarray.scalar"),
    np.dtype, np.ndarray,
    np.dtypes.Float64DType, np.dtypes.Float32DType,
    np.dtypes.Int64DType, np.dtypes.Int32DType,
    np.dtypes.UInt8DType, np.dtypes.BoolDType,
])

ckpt = torch.load(CKPT_PATH, map_location="cpu")
print("Top-level keys:", list(ckpt.keys()))

# Ark+ stores teacher EMA weights under 'teacher'
state = ckpt.get("teacher", ckpt)
state_keys = list(state.keys())
print("State dict keys (first 10):", state_keys[:10])
print("Total keys:", len(state_keys))

## 3. Load `OmniSwinTransformer`

Ark+ extends `timm.SwinTransformer` with:
- `generate_embeddings(x)` — forward through backbone + projector → [B, 1376]
- `omni_heads` — per-dataset classification heads (not needed for embedding extraction)

In [ ]:
import os, sys, shutil, subprocess

# Ark+'s Swin checkpoint was saved with timm==0.5.4, whose Swin layout
# differs from modern timm (downsample placement). Install 0.5.4 into a
# side-by-side directory so the main venv keeps its newer timm for the
# other backbones (EVA-X needs >=0.9), and prepend the legacy path here.
#
# `--no-deps` is critical: without it, uv pulls timm 0.5.4's old
# torchvision into the same target dir, sys.path picks it up ahead of
# the venv's torchvision, and you get a torch/torchvision ABI clash
# ("operator torchvision::nms does not exist").
LEGACY_TIMM = "../../third_party_models/Ark/timm-054"

# Re-install if missing OR if a previous (broken) install bundled
# torchvision into the target dir.
_needs_install = (
    not os.path.exists(LEGACY_TIMM)
    or os.path.exists(os.path.join(LEGACY_TIMM, "torchvision"))
)
if _needs_install:
    if os.path.exists(LEGACY_TIMM):
        shutil.rmtree(LEGACY_TIMM)
    subprocess.run(
        ["uv", "pip", "install",
         "--target", LEGACY_TIMM,
         "--python", sys.executable,
         "--no-deps",
         "timm==0.5.4"],
        check=True,
    )

# Drop any modern-timm imports so the fresh import binds to 0.5.4.
for m in list(sys.modules):
    if m == "timm" or m.startswith("timm."):
        del sys.modules[m]
sys.path.insert(0, LEGACY_TIMM)

import timm
from timm.models.swin_transformer import SwinTransformer
print("timm version (should be 0.5.4):", timm.__version__)

# Official loading recipe from https://github.com/jlianglab/Ark
# (Ark_Plus README) — bare timm SwinTransformer, head deleted below.
model = SwinTransformer(
    num_classes=1,
    img_size=768,
    patch_size=4,
    window_size=12,
    embed_dim=192,
    depths=(2, 2, 18, 2),
    num_heads=(6, 12, 24, 48),
)

# Strip "module." prefix from the teacher state dict.
state_dict = {k.replace("module.", ""): v for k, v in state.items()}

# attn_mask buffers depend on input size; head is per-task. Drop both.
k_del = [k for k in state_dict if "attn_mask" in k] + ["head.weight", "head.bias"]
for k in k_del:
    state_dict.pop(k, None)
print(f"Removed {len(k_del)} keys (attn_mask + head)")

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Missing (non-head):", [k for k in missing if "head" not in k][:5])
print("Unexpected:        ", unexpected[:5])

model.eval()
print("Swin-L num_features:", model.num_features)   # 1536

## 4. Image transform

Ark+ uses 768×768 input with ImageNet normalization and RGB (grayscale CXR → repeat to 3ch).

In [ ]:
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from PIL import Image

INPUT_SIZE = 768

def make_transform():
    return transforms.Compose([
        transforms.Resize((INPUT_SIZE, INPUT_SIZE), interpolation=InterpolationMode.BICUBIC),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.repeat(3, 1, 1)),   # 1ch → 3ch
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225],
        ),
    ])

transform = make_transform()

def preprocess(path: str) -> torch.Tensor:
    """Return [1, 3, 768, 768] batch tensor."""
    img = Image.open(path)
    return transform(img).unsqueeze(0)

# Smoke-test: 768×768 images are large — ~1.8M elements per image
dummy = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE)
print("dummy input:", dummy.shape, f"({dummy.numel() / 1e6:.1f}M elements)")

## 5. Extract embeddings

In [ ]:
# Note: 768px Swin-L is memory-heavy — use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
dummy = dummy.to(device)

with torch.no_grad():
    # Raw Swin-L pooled features (1536-dim) — drop the head and read
    # forward_features directly.
    emb = model.forward_features(dummy)   # [B, 1536]

print("embedding:", emb.shape)            # [1, 1536]
print("range:", emb.min().item(), emb.max().item())

## Segmentation mode — spatial feature map (Swin)

Ark+ uses **Swin Transformer Large** — a hierarchical architecture with no flat patch
sequence. The final-stage output is a spatial feature map `[B, C, H, W]` available
via a hook on `model.norm` (Swin's final LayerNorm before adaptive avg pool).

For 768×768 input: Swin-L stride is 32 → **24×24** spatial grid, C=1536.
This is analogous to ViT patch features at 32-px stride.

In [ ]:
import math

# Swin Transformer is hierarchical — the final stage outputs [B, H*W, C] before avg pool.
# Capture it via a forward hook on the model's top-level norm layer.
last_stage_feat = {}

def _hook(module, inp, out):
    last_stage_feat["feat"] = out   # [B, H*W, C] — Swin final stage output

handle = model.norm.register_forward_hook(_hook)
with torch.no_grad():
    _ = model.forward_features(dummy)
handle.remove()

feat = last_stage_feat["feat"]   # [B, H*W, C] — flattened spatial map
print("Swin last-stage output:", feat.shape)
# For 768px Swin-L: [1, 576, 1536]  (768 / 32 = 24, 24×24=576 tokens, C=1536)

B, HW, C = feat.shape
H = W = int(math.sqrt(HW))
assert H * W == HW, f"Swin spatial grid {HW} is not a perfect square"

spatial = feat.permute(0, 2, 1).reshape(B, C, H, W)
print(f"spatial map: {spatial.shape}")   # [1, 1536, 24, 24] for 768px Swin-L

## 6. Real CXR — end-to-end

In [ ]:
CXR_PATH = "/path/to/cxr.jpg"

x = preprocess(CXR_PATH).to(device)
print("input:", x.shape, "range:", x.min().item(), x.max().item())

with torch.no_grad():
    emb = model.forward_features(x)   # [1, 1536]

print("embedding:", emb.shape)
print("L2 norm:  ", emb.norm(dim=-1).item())

## Notes for RadHarmony integration

```
model_call : lambda m, x: m.forward_features(x)
pool       : None
embed_dim  : 1536  (raw Swin-L features)
transform  : Resize(768,768,BICUBIC) → Grayscale → ToTensor
             → repeat(3ch) → Normalize(ImageNet)
```

**Loading recipe** (from the official Ark+ README):
- Build a bare `timm.models.swin_transformer.SwinTransformer`
- Strip `module.` prefix from `checkpoint["teacher"]`
- Drop `attn_mask` buffers (input-size-dependent) and `head.{weight,bias}` (per-task)
- `load_state_dict(strict=False)`

**timm version**: `timm==0.5.4` is required. This conflicts with EVA-X (`timm==0.9.0`).
Use separate venvs or conda envs for these two backbones.

**Memory**: 768px input is 4× the pixel count of 384px. Expect ~2–4 GB GPU memory
per image at float32; reduce `batch_size` in `BaseClsEvaluator` accordingly (try 8–16).

**Weights**: The Google Form request is manual — weights are not auto-downloadable,
so this backbone cannot be wired into a factory that downloads on first call.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- Hook `model.norm` to get Swin's final-stage spatial output `[B, H*W, C]`
- Spatial map: reshape to `[B, 1536, 24, 24]` for 768px Swin-L (stride 32 → 24×24)

In [ ]:
import time

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

batch = torch.zeros(4, 3, INPUT_SIZE, INPUT_SIZE, device=device)   # batch of 4
t0 = time.perf_counter()
with torch.no_grad():
    out = model.forward_features(batch)
elapsed = time.perf_counter() - t0

print(f"batch={batch.shape}  →  {out.shape}  in {elapsed:.2f}s")
if torch.cuda.is_available():
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    print(f"peak GPU memory: {peak_gb:.2f} GB")

## Notes for RadHarmony integration

```
model_call : lambda m, x: m.forward_features(x)
pool       : None
embed_dim  : 1536  (raw Swin-L features)
transform  : Resize(768,768,BICUBIC) → Grayscale → ToTensor
             → repeat(3ch) → Normalize(ImageNet)
```

**timm version**: `timm==0.5.4` is required. This conflicts with EVA-X (`timm==0.9.0`).
Use separate venvs or conda envs for these two backbones.

**Memory**: 768px input is 4× the pixel count of 384px. Expect ~2–4 GB GPU memory
per image at float32; reduce `batch_size` in `BaseClsEvaluator` accordingly (try 8–16).

**Weights**: The Google Form request is manual — weights are not auto-downloadable,
so this backbone cannot be wired into a factory that downloads on first call.